# Team 21 - EuroSAT label-efficiency: multispectral / band ablation
Settings: **Accelerator = GPU**, **Internet = On**. Results go to `results/runs_ms.csv`.
Inputs compared (all from the 13-band files): `rgb_ms`, `rgb+nir`, `rgb+nir+rededge`, `rgb+nir+swir`, `all13`.

In [ ]:
# 1) get the code from GitHub (public repo)
!git clone https://github.com/kumarAyush24029/Eurosat-label-efficiency.git /kaggle/working/proj
%cd /kaggle/working/proj
!pip install -q timm tifffile imagecodecs

In [ ]:
# 2) download 13-band EuroSAT (~2 GB; takes a few minutes)
!mkdir -p data && cd data && wget -q -O ms.zip 'https://zenodo.org/records/7711810/files/EuroSAT_MS.zip?download=1' && unzip -q -n ms.zip -d raw_ms
!ls data/raw_ms
!ls data/raw_ms/* | head

In [ ]:
# 3) sanity run on the 13-band loader (the folder name below must match the `ls` output above)
!python src/train.py --ms_root data/raw_ms/EuroSAT_MS --splits data/splits.json --out results/smoke_ms.csv --models cnn --inputs all13 --fracs 0.01 --seeds 0 --epochs 2

## Band-ablation grid (ResNet-50, ImageNet init) x 6 label fractions x 3 seeds
Kaggle gives 2 GPUs per session but one training process uses only one, so we start TWO processes at once (one per GPU), each with its own results file. Cell 3 above must have run first: it builds the data cache that both processes read.

In [ ]:
%%bash
ARGS="--ms_root data/raw_ms/EuroSAT_MS --splits data/splits.json --fracs 0.01 0.02 0.05 0.1 0.25 1.0 --seeds 0 1 2 --models resnet50 --inits imagenet --epochs 20"
# GPU 0: three input settings; GPU 1: the other two. `sed -u` tags each log line with its GPU.
CUDA_VISIBLE_DEVICES=0 python src/train.py $ARGS --out results/runs_ms_a.csv --inputs rgb_ms all13 rgb+nir 2>&1 | sed -u 's/^/[GPU0] /' &
CUDA_VISIBLE_DEVICES=1 python src/train.py $ARGS --out results/runs_ms_b.csv --inputs rgb+nir+rededge rgb+nir+swir 2>&1 | sed -u 's/^/[GPU1] /' &
wait
echo ALL DONE

In [ ]:
# 4) bundle to download (Output tab -> results_ms.zip)
!python src/analyze.py --csv results/runs_ms_a.csv results/runs_ms_b.csv --out figures
!python src/eda.py --ms_root data/raw_ms/EuroSAT_MS --out figures
!zip -qr results_ms.zip results figures && ls -la results_ms.zip